<a href="https://colab.research.google.com/github/tishasetia13/inside-out-emotion-classifier/blob/main/inside_out.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###GETTING OUR DATASET READY

In [1]:
import pandas as pd

url = "https://huggingface.co/datasets/gsri-18/ISEAR-dataset-complete/resolve/main/ISEAR_dataset_complete.csv"
df = pd.read_csv(url)

In [2]:
#df.head()
#df.shape
#df["emotion"].value_counts()
#df.isnull().sum()

In [3]:
df = df[df['emotion'].isin(['joy', 'sadness', 'anger', 'fear', 'disgust'])]
df = df.drop(columns=['Unnamed: 2'])

In [4]:
#df.shape
#df['emotion'].value_counts()

In [5]:
df['content'] = df['content'].str.replace('\\n', ' ', regex=False)
df['content'] = df['content'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [6]:
#pd.set_option('display.max_colwidth', None)
#df.sample(20, random_state=42)['content']
#df['content'].str.len().describe()

In [7]:
df = df.reset_index(drop=True)

In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['emotion'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['emotion'], random_state=42)

print(train_df.shape, val_df.shape, test_df.shape)

(3776, 2) (809, 2) (810, 2)


###MAKING TOKENIZER AND DATA BATCHING

In [9]:
from transformers import AutoTokenizer
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# sample = 'Hi, my name is Tisha'
# encoded = tokenizer(sample)
# print(encoded)
# print(tokenizer.convert_ids_to_tokens(encoded['input_ids']))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
# lengths = [len(tokenizer.encode(text)) for text in train_df['content']]

# print(pd.Series(lengths).describe())
# print("95th percentile:", pd.Series(lengths).quantile(0.95))
# print("99th percentile:", pd.Series(lengths).quantile(0.99))

In [11]:
# batch_texts = train_df['content'].tolist()[:4]

# batch = tokenizer(
#     batch_texts,
#     padding='max_length',
#     truncation=True,
#     max_length=64,
#     return_tensors='pt',
# )

# print(batch["input_ids"].shape)
# print(batch["attention_mask"].sum(dim=1))
# print(batch["input_ids"][0])

In [12]:
labels = sorted(train_df['emotion'].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# print(label2id)
# print(id2label)

In [13]:
for df in [train_df, val_df, test_df]:
  df['label'] = df['emotion'].map(label2id)

# print(train_df[["emotion", "label"]].head())
# print(train_df["label"].value_counts().sort_index())
# print("Missing labels:", train_df["label"].isna().sum(), val_df["label"].isna().sum(), test_df["label"].isna().sum())

In [14]:
import torch

def encode_split(df):
  enc = tokenizer(
      df['content'].tolist(),
      padding='max_length',
      truncation=True,
      max_length=64,
      return_tensors='pt',
  )
  enc['labels'] = torch.tensor(df['label'].tolist())
  return enc

train_enc = encode_split(train_df)
val_enc = encode_split(val_df)
test_enc = encode_split(test_df)

# for name, enc in [("train", train_enc), ("val", val_enc), ("test", test_enc)]:
#     print(name, enc["input_ids"].shape, enc["attention_mask"].shape, enc["labels"].shape)
# print(train_enc["labels"][:5])

In [15]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(enc, batch_size, shuffle):
  dataset = TensorDataset(enc['input_ids'], enc['attention_mask'], enc['labels'])
  return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_enc, batch_size=16, shuffle=True)
val_loader = make_loader(val_enc, batch_size=32, shuffle=False)
test_loader = make_loader(test_enc, batch_size=32, shuffle=False)

# print("Batches per epoch (train):", len(train_loader))
# ids, mask, y = next(iter(train_loader))
# print(ids.shape, mask.shape, y.shape)
# print(y)

In [16]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

needed = ["tokenizer", "label2id", "id2label", "train_enc", "val_enc",
          "test_enc", "train_loader", "val_loader", "test_loader"]
for name in needed:
    print(f"{name:14s}", "OK" if name in globals() else "MISSING")

GPU available: True
GPU name: Tesla T4
Using device: cuda
tokenizer      OK
label2id       OK
id2label       OK
train_enc      OK
val_enc        OK
test_enc       OK
train_loader   OK
val_loader     OK
test_loader    OK


In [17]:
from transformers import AutoModelForSequenceClassification

model_name='distilbert-base-uncased'

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
)

model = model.to(device)

print('Model Class: ', type(model).__name__)
print('Model is on: ', next(model.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Class:  DistilBertForSequenceClassification
Model is on:  cuda:0


In [18]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()
print(model)

Total parameters:     66,957,317
Trainable parameters: 66,957,317

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        

In [19]:
batch = next(iter(train_loader))

print("Type of batch:", type(batch))
print("Number of items:", len(batch))
for i, v in enumerate(batch):
    print(f"item {i}: shape={tuple(v.shape)}  dtype={v.dtype}  device={v.device}")

Type of batch: <class 'list'>
Number of items: 3
item 0: shape=(16, 64)  dtype=torch.int64  device=cpu
item 1: shape=(16, 64)  dtype=torch.int64  device=cpu
item 2: shape=(16,)  dtype=torch.int64  device=cpu


In [20]:
input_ids, attention_mask, labels = batch

print("input_ids[0], first 15:     ", input_ids[0][:15].tolist())
print("attention_mask[0], first 15:", attention_mask[0][:15].tolist())
print("unique values in item 1:    ", torch.unique(attention_mask).tolist())
print("labels:", labels.tolist())
print()
print("Decoded example 0:")
print(tokenizer.decode(input_ids[0], skip_special_tokens=False))

input_ids[0], first 15:      [101, 2293, 3471, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask[0], first 15: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
unique values in item 1:     [0, 1]
labels: [4, 4, 3, 4, 3, 4, 2, 0, 1, 4, 0, 2, 2, 0, 0, 3]

Decoded example 0:
[CLS] love problems. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [23]:
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
labels = labels.to(device)

model.eval()
with torch.no_grad():
  outputs = model(input_ids=input_ids,
                      attention_mask=attention_mask,
                      labels=labels)

print("logits shape:", tuple(outputs.logits.shape))
print("loss:", outputs.loss.item())
print("expected loss for a clueless model (ln 5): 1.609")
print()
print("logits for example 0:", outputs.logits[0].tolist())
print("predicted class ids: ", outputs.logits.argmax(dim=-1).tolist())
print("true labels:         ", labels.tolist())

logits shape: (16, 5)
loss: 1.6378695964813232
expected loss for a clueless model (ln 5): 1.609

logits for example 0: [0.05386905372142792, 0.1390921175479889, -0.165388822555542, -0.08767420798540115, 0.009594634175300598]
predicted class ids:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
true labels:          [4, 4, 3, 4, 3, 4, 2, 0, 1, 4, 0, 2, 2, 0, 0, 3]


In [25]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay: ", optimizer.param_groups[0]["weight_decay"])

Optimizer: AdamW
Learning rate: 2e-05
Weight decay:  0.01


In [26]:
model.eval()
with torch.no_grad():
  loss_before = model(input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels).loss.item()
w_before = model.classifier.weight.detach().clone()

model.train()
optimizer.zero_grad()
outputs = model(input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels)
loss = outputs.loss
loss.backward()
grad_size = model.classifier.weight.grad.norm().item()
optimizer.step()

model.eval()
with torch.no_grad():
    loss_after = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels).loss.item()

change = (model.classifier.weight.detach() - w_before).abs().max().item()

print(f"loss before step:            {loss_before:.4f}")
print(f"loss during step (train):    {loss.item():.4f}")
print(f"loss after step:             {loss_after:.4f}")
print(f"gradient size (classifier):  {grad_size:.4f}")
print(f"biggest weight change:       {change:.8f}")

loss before step:            1.6379
loss during step (train):    1.6344
loss after step:             1.6149
gradient size (classifier):  1.1739
biggest weight change:       0.00002001


In [27]:
num_epochs = 3

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0

  for batch in train_loader:
    input_ids, attention_mask, labels = [t.to(device) for t in batch]

    optimizer.zero_grad()
    outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm(model.parameters(), 1.0)
    optimizer.step()

    running_loss += loss.item()

  avg_loss = running_loss / len(train_loader)
  print(f"Epoch {epoch + 1}/{num_epochs}  train loss: {avg_loss:.4f}")

/tmp/ipykernel_1162/332502869.py:16: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(model.parameters(), 1.0)


Epoch 1/3  train loss: 1.0372
Epoch 2/3  train loss: 0.5467
Epoch 3/3  train loss: 0.3559


In [30]:
model.eval()
val_loss_total = 0.0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids, attention_mask, labels = [t.to(device) for t in batch]

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        val_loss_total += outputs.loss.item()
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

val_loss = val_loss_total / len(val_loader)
correct = sum(p == l for p, l in zip(all_preds, all_labels))
val_acc = correct / len(all_labels)

print(f"Validation loss:     {val_loss:.4f}")
print(f"Validation accuracy: {val_acc:.4f}  ({correct}/{len(all_labels)})")

Validation loss:     0.7712
Validation accuracy: 0.7565  (612/809)
